# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raju-cse/Flyrank-internship_ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

. Method choice and why
This lane is Refresh / Content Opportunity Scoring. The target is is_declining_label, defined as trend_direction == 'down'.

I will compare Logistic Regression, a shallow Decision Tree, and a constrained Random Forest. I will select the model using Precision@50, because the practical task is to rank a small review queue and put useful pages near the top. I will also report ROC-AUC, average precision, precision, recall, and F1.

The models are deliberately limited in complexity so that a more complex model is only preferred when it produces a useful improvement.

In [6]:
!git clone https://github.com/raju-cse/Flyrank-internship_ml.git
%cd Flyrank-internship_ml

Cloning into 'Flyrank-internship_ml'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 138 (delta 49), reused 98 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 1.87 MiB | 18.98 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/Flyrank-internship_ml/Flyrank-internship_ml


In [7]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, f1_score,
    precision_score, recall_score, roc_auc_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
ROOT = Path.cwd()
FEATURE_PATH = ROOT / "data" / "processed" / "refresh_feature_vector.csv"
BASELINE_PATH = ROOT / "data" / "processed" / "baseline_refresh_queue.csv"

print("Working directory:", ROOT)
print("Feature file:", FEATURE_PATH)
print("Baseline file:", BASELINE_PATH)

Working directory: /content/Flyrank-internship_ml/Flyrank-internship_ml
Feature file: /content/Flyrank-internship_ml/Flyrank-internship_ml/data/processed/refresh_feature_vector.csv
Baseline file: /content/Flyrank-internship_ml/Flyrank-internship_ml/data/processed/baseline_refresh_queue.csv


## 2. Split design

A client-aware holdout is preferred because multiple content rows can belong to the same client. Holding out whole clients reduces the risk that the model learns client-specific patterns that make a row-level random split look better than it really is.

The split uses a fixed random seed (42) and approximately 20% of clients for testing. If there are too few clients or the holdout loses one target class, the code falls back to a stratified row holdout.

In [8]:
# Create the prepared feature files if they are not already present.
if not FEATURE_PATH.exists() or not BASELINE_PATH.exists():
    import subprocess, sys
    subprocess.run([sys.executable, "scripts/01_prepare_features.py"], check=True)
    subprocess.run([sys.executable, "scripts/02_baseline_score.py"], check=True)

frame = pd.read_csv(FEATURE_PATH)
baseline_frame = pd.read_csv(BASELINE_PATH)

MODEL_NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d",
    "log_ai_sessions_90d", "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct"
]

MODEL_CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier"
]

def build_feature_matrix(df):
    numeric = [c for c in MODEL_NUMERIC_FEATURES if c in df.columns]
    categorical = [c for c in MODEL_CATEGORICAL_FEATURES if c in df.columns]

    n = (
        df[numeric]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )
    c = df[categorical].fillna("unknown").astype(str)
    encoded = pd.get_dummies(c, prefix=categorical, dtype=float)

    return pd.concat(
        [n.reset_index(drop=True), encoded.reset_index(drop=True)],
        axis=1
    )

X = build_feature_matrix(frame)
y = frame["is_declining_label"].astype(int)

all_idx = np.arange(len(frame))
clients = frame["client_id"].fillna("unknown").astype(str)
unique_clients = clients.drop_duplicates().to_numpy()

if len(unique_clients) >= 5:
    rng = np.random.default_rng(RANDOM_STATE)
    shuffled = rng.permutation(unique_clients)
    test_client_count = max(1, int(round(len(shuffled) * 0.20)))
    test_clients = set(shuffled[:test_client_count])

    test_mask = clients.isin(test_clients).to_numpy()
    train_idx = all_idx[~test_mask]
    test_idx = all_idx[test_mask]

    if (
        len(train_idx) > 0
        and len(test_idx) > 0
        and y.iloc[train_idx].nunique() == 2
        and y.iloc[test_idx].nunique() == 2
    ):
        split_strategy = "client_holdout"
    else:
        train_idx, test_idx = train_test_split(
            all_idx, test_size=0.20,
            random_state=RANDOM_STATE, stratify=y
        )
        split_strategy = "stratified_row_holdout"
else:
    train_idx, test_idx = train_test_split(
        all_idx, test_size=0.20,
        random_state=RANDOM_STATE, stratify=y
    )
    split_strategy = "stratified_row_holdout"

train_idx = np.asarray(train_idx)
test_idx = np.asarray(test_idx)

print("Split strategy:", split_strategy)
print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Train positive rate:", round(y.iloc[train_idx].mean(), 3))
print("Test positive rate:", round(y.iloc[test_idx].mean(), 3))

Split strategy: client_holdout
Train rows: 27675
Test rows: 2325
Train positive rate: 0.555
Test positive rate: 0.391


## 3. Train + compare vs my baseline

The baseline is the deterministic refresh score from Week 4. The comparison below uses the same test rows and the same metrics for the baseline and every learned model.

Primary metric: Precision@50, because this is a ranked review problem.

In [9]:
def precision_at_k(y_true, scores, k=50):
    temp = pd.DataFrame({
        "y": np.asarray(y_true),
        "score": np.asarray(scores)
    })
    top = temp.sort_values("score", ascending=False).head(min(k, len(temp)))
    return float(top["y"].mean()) if len(top) else 0.0

def metrics(y_true, scores):
    pred = (np.asarray(scores) >= 0.5).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "ROC AUC": roc_auc_score(y_true, scores),
        "Average Precision": average_precision_score(y_true, scores),
        "Precision@20": precision_at_k(y_true, scores, 20),
        "Precision@50": precision_at_k(y_true, scores, 50),
        "Precision@100": precision_at_k(y_true, scores, 100),
    }

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ]),
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        max_depth=5,
        min_samples_leaf=50,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        class_weight="balanced_subsample",
        max_depth=10,
        min_samples_leaf=25,
        n_estimators=200,
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
}

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

baseline_lookup = baseline_frame.set_index("content_id")["baseline_refresh_score"]
baseline_scores = (
    frame.iloc[test_idx]["content_id"]
    .map(baseline_lookup)
    .fillna(0)
    .to_numpy()
)

results = {"Week-4 baseline": metrics(y_test, baseline_scores)}
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    scores = model.predict_proba(X_test)[:, 1]
    results[name] = metrics(y_test, scores)

comparison = pd.DataFrame(results).T.sort_values(
    "Precision@50", ascending=False
)

display(comparison.round(3))

best_model_name = comparison.index[0]
print("Selected model:", best_model_name)
print("Selection metric: Precision@50")

,Accuracy,Precision,Recall,F1,ROC AUC,Average Precision,Precision@20,Precision@50,Precision@100
Random Forest,0.672,0.561,0.744,0.640,0.750,0.618,0.65,0.74,0.72
Decision Tree,0.677,0.569,0.716,0.634,0.742,0.575,0.65,0.66,0.68
Logistic Regression,0.661,0.566,0.567,0.566,0.700,0.522,0.35,0.40,0.44
Week-4 baseline,0.609,0.499,0.189,0.274,0.627,0.468,0.15,0.24,0.36


Selected model: Random Forest
Selection metric: Precision@50


### Result interpretation

On the client-held-out test set, Random Forest achieved the best Precision@50 of 0.74, compared with 0.24 for the Week-4 baseline. It also achieved a ROC-AUC of 0.750, compared with 0.627 for the baseline. Decision Tree achieved a Precision@50 of 0.66, while Logistic Regression achieved 0.40.

Based on the primary metric, Random Forest was selected because it provided the strongest improvement over the Week-4 baseline. The result suggests that the learned model can rank potentially declining content more effectively than the baseline rule on the same test set. However, the model should be used as decision support for manual review, not as an automatic decision-making system.

## 4. Errors and interpretation

A useful error analysis should show both false positives and false negatives rather than only a large metric table. I will inspect the highest-confidence mistakes and then examine the features used most strongly by the selected model.

In [10]:
best_model = fitted_models[best_model_name]
test_scores = best_model.predict_proba(X_test)[:, 1]

error_frame = frame.iloc[test_idx][[
    "content_id", "client_id", "is_declining_label", "trend_direction",
    "impressions_90d", "sessions_90d", "avg_position", "ctr",
    "word_count", "days_since_last_update"
]].copy()

error_frame["predicted_probability"] = test_scores

error_frame["error_type"] = np.where(
    (error_frame["is_declining_label"] == 0) &
    (error_frame["predicted_probability"] >= 0.5),
    "false_positive",
    np.where(
        (error_frame["is_declining_label"] == 1) &
        (error_frame["predicted_probability"] < 0.5),
        "false_negative",
        "correct"
    )
)

print("Error counts:")
display(error_frame["error_type"].value_counts().to_frame("count"))

print("Highest-confidence false positives:")
display(
    error_frame[error_frame.error_type == "false_positive"]
    .sort_values("predicted_probability", ascending=False)
    .head(10)
)

print("Highest-confidence false negatives:")
display(
    error_frame[error_frame.error_type == "false_negative"]
    .sort_values("predicted_probability", ascending=True)
    .head(10)
)

if isinstance(best_model, Pipeline):
    estimator = best_model.named_steps["model"]
    importance_values = np.abs(estimator.coef_[0])
elif hasattr(best_model, "feature_importances_"):
    importance_values = best_model.feature_importances_
else:
    importance_values = np.zeros(X.shape[1])

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": importance_values
}).sort_values("importance", ascending=False).head(15)

print("Top features:")
display(importance)

Error counts:


,count
error_type,
correct,1563
false_positive,529
false_negative,233


Highest-confidence false positives:


,content_id,client_id,is_declining_label,trend_direction,impressions_90d,sessions_90d,avg_position,ctr,word_count,days_since_last_update,predicted_probability,error_type
23250,content_d2dffcc697a4,client_f74efabef1,0,stable,5091,30,14.1,0.20,4496.0,20,0.737130,false_positive
23559,content_00603b0349b4,client_f74efabef1,0,up,1076,9,25.6,0.09,2439.0,20,0.734944,false_positive
25913,content_331182ca4cae,client_f74efabef1,0,up,3026,24,35.9,0.00,3546.0,20,0.733631,false_positive
23750,content_e55b8ab078b0,client_f74efabef1,0,stable,369,2,21.8,0.00,2192.0,20,0.733059,false_positive
10155,content_643f585dc7f7,client_f74efabef1,0,up,761,6,25.1,0.39,1980.0,20,0.731120,false_positive
5966,content_f5013794ba57,client_f74efabef1,0,new,881,2,15.7,0.00,3622.0,20,0.730532,false_positive
28337,content_ea4417d89e2c,client_f74efabef1,0,stable,352,4,11.9,0.00,2556.0,20,0.729056,false_positive
4249,content_db1cd41b4b4f,client_f74efabef1,0,up,1482,19,12.9,0.00,2221.0,105,0.729052,false_positive
21530,content_b15a8dbdf66f,client_f74efabef1,0,up,1647,10,22.4,0.18,4095.0,20,0.727853,false_positive
2380,content_96da95476e63,client_f74efabef1,0,stable,784,6,7.4,0.00,2598.0,20,0.724807,false_positive


Highest-confidence false negatives:


,content_id,client_id,is_declining_label,trend_direction,impressions_90d,sessions_90d,avg_position,ctr,word_count,days_since_last_update,predicted_probability,error_type
5770,content_28b4223f4e5f,client_98a3ab7c34,1,down,1,1,0.0,0.00,3109.0,1,0.079867,false_negative
3879,content_34b14c00f80c,client_d4735e3a26,1,down,3,1,0.0,0.00,659.0,20,0.082196,false_negative
27177,content_79ac977c6e0b,client_f74efabef1,1,down,3,1,0.7,0.00,2304.0,8,0.149546,false_negative
22991,content_472ce7ae14c0,client_d4735e3a26,1,down,3,2,0.3,33.33,684.0,20,0.152184,false_negative
5608,content_a55d958ec725,client_d4735e3a26,1,down,3,1,2.7,0.00,837.0,20,0.163987,false_negative
12864,content_f1ef151d5e36,client_d4735e3a26,1,down,3,2,2.0,0.00,978.0,20,0.165946,false_negative
12076,content_230de4c50860,client_d4735e3a26,1,down,3,1,2.0,0.00,824.0,20,0.169816,false_negative
25838,content_cbc3b52a2ac1,client_98a3ab7c34,1,down,2,1,3.0,0.00,2286.0,1,0.171345,false_negative
13659,content_4c437dd8c1ee,client_d4735e3a26,1,down,3,1,3.0,0.00,840.0,20,0.174066,false_negative
23810,content_37804210415c,client_d4735e3a26,1,down,4,3,2.0,0.00,813.0,20,0.174828,false_negative


Top features:


,feature,importance
9,days_with_impressions,0.134951
5,log_impressions_90d,0.129377
14,avg_position,0.109203
11,content_age_days,0.092048
4,char_count,0.038676
32,age_tier_365+,0.036847
6,log_clicks_90d,0.036572
3,word_count,0.035406
13,ctr,0.035156
16,scroll_rate,0.033876


### What the errors mean in practice

The model should be treated as a **review-ranking aid**, not an automatic publishing decision.

False positives are pages that look risky from the available signals but are not labeled as declining. They can still be reasonable manual-review candidates. False negatives are declining pages that the model ranks too low, showing that the available features do not capture every reason a page can lose performance.

The strongest features should therefore be interpreted as **associations in this dataset**, not causal explanations.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.